# 📝 Lección 8: Text y Tex en Manim

## Contenido de esta lección:
1. **Clase Text** - Renderizado de texto sin LaTeX (Pango)
2. **Estilos de Text** - t2c, t2f, t2s, t2w, gradientes
3. **Indexación de Text** - Acceso a caracteres y slicing
4. **Clase MathTex** - Fórmulas matemáticas con LaTeX
5. **Aislamiento de partes** - substrings_to_isolate y doble llave `{{}}`
6. **Métodos de MathTex** - get_part_by_tex, set_color_by_tex
7. **Clase Tex** - LaTeX en modo texto normal
8. **⭐ TRANSFORMACIONES DE TEXTO** (Sección extendida)
   - TransformMatchingTex
   - TransformMatchingShapes
   - key_map, fade_transform_mismatches, transform_mismatches
   - ReplacementTransform vs Transform
9. **Animaciones de creación** - Write, AddTextLetterByLetter, TypeWithCursor
10. **Ejemplos prácticos combinados**
11. **Ejercicios**

---

In [ ]:
from manim import *
import numpy as np

---
## 1. 📖 Clase Text - Texto sin LaTeX

La clase `Text` renderiza texto usando **Pango** (no LaTeX). Es ideal para:
- Títulos y texto descriptivo
- Texto que no requiere símbolos matemáticos
- Fuentes personalizadas del sistema

### Parámetros principales:
| Parámetro | Descripción | Default |
|-----------|-------------|---------|
| `font_size` | Tamaño de fuente | 48 |
| `color` | Color del texto | WHITE |
| `font` | Nombre de fuente del sistema | "" |
| `slant` | Inclinación (NORMAL, ITALIC) | NORMAL |
| `weight` | Peso (NORMAL, BOLD) | NORMAL |
| `line_spacing` | Espaciado entre líneas | -1 |
| `t2c` | Dict texto→color | {} |
| `t2f` | Dict texto→fuente | {} |
| `t2s` | Dict texto→inclinación | {} |
| `t2w` | Dict texto→peso | {} |
| `gradient` | Tuple de colores para gradiente | None |

In [ ]:
%%manim -qm -v WARNING TextBasico

class TextBasico(Scene):
    def construct(self):
        # Texto simple
        texto1 = Text("¡Hola Manim!")
        
        # Texto con tamaño personalizado
        texto2 = Text("Texto más grande", font_size=72)
        texto2.next_to(texto1, DOWN)
        
        # Texto con color
        texto3 = Text("Texto colorido", color=YELLOW)
        texto3.next_to(texto2, DOWN)
        
        # Texto con fuente del sistema (si está instalada)
        texto4 = Text("Fuente Arial", font="Arial", font_size=36)
        texto4.next_to(texto3, DOWN)
        
        grupo = VGroup(texto1, texto2, texto3, texto4).arrange(DOWN, buff=0.5)
        
        self.play(Write(texto1))
        self.play(Write(texto2))
        self.play(Write(texto3))
        self.play(Write(texto4))
        self.wait()

---
## 2. 🎨 Estilos de Text - t2c, t2f, t2s, t2w

Los diccionarios `t2*` permiten aplicar estilos a **partes específicas** del texto:

- **t2c** (text to color): Colorea partes específicas
- **t2f** (text to font): Cambia la fuente de partes específicas
- **t2s** (text to slant): Aplica cursiva (ITALIC, NORMAL)
- **t2w** (text to weight): Aplica negrita (BOLD, NORMAL)

### Sintaxis:
```python
Text("texto completo", t2c={"palabra": COLOR})
```

También puedes usar **slicing** con índices:
```python
Text("Hola", t2c={"[0:2]": RED})  # Colorea "Ho"
```

In [ ]:
%%manim -qm -v WARNING TextEstilos

class TextEstilos(Scene):
    def construct(self):
        # t2c - Text to Color
        texto_color = Text(
            "Python es genial",
            t2c={"Python": BLUE, "genial": GREEN}
        )
        
        # t2c con índices (slicing)
        texto_slice = Text(
            "ABCDEFGH",
            t2c={"[0:3]": RED, "[3:6]": YELLOW, "[6:]": BLUE}
        )
        texto_slice.next_to(texto_color, DOWN, buff=0.5)
        
        # t2s - Text to Slant (cursiva)
        texto_cursiva = Text(
            "Texto con énfasis",
            t2s={"énfasis": ITALIC}
        )
        texto_cursiva.next_to(texto_slice, DOWN, buff=0.5)
        
        # t2w - Text to Weight (negrita)
        texto_negrita = Text(
            "Palabra importante aquí",
            t2w={"importante": BOLD},
            t2c={"importante": YELLOW}
        )
        texto_negrita.next_to(texto_cursiva, DOWN, buff=0.5)
        
        # Combinación de todos
        texto_combo = Text(
            "Manim es increíble",
            t2c={"Manim": BLUE, "increíble": GREEN},
            t2w={"Manim": BOLD},
            t2s={"increíble": ITALIC}
        )
        texto_combo.next_to(texto_negrita, DOWN, buff=0.5)
        
        grupo = VGroup(texto_color, texto_slice, texto_cursiva, 
                       texto_negrita, texto_combo).move_to(ORIGIN)
        
        for texto in grupo:
            self.play(Write(texto), run_time=0.8)
        self.wait()

### 2.1 Gradientes en Text

Puedes aplicar un **gradiente de colores** a todo el texto:

In [ ]:
%%manim -qm -v WARNING TextGradiente

class TextGradiente(Scene):
    def construct(self):
        # Gradiente simple (2 colores)
        texto1 = Text("Gradiente Arcoíris", font_size=48)
        texto1.set_color_by_gradient(RED, ORANGE, YELLOW, GREEN, BLUE, PURPLE)
        
        # Gradiente con parámetro gradient en constructor
        texto2 = Text(
            "Fuego y Hielo",
            font_size=48,
            gradient=(RED, ORANGE, YELLOW)
        )
        texto2.next_to(texto1, DOWN, buff=0.8)
        
        # Gradiente vertical usando set_color_by_gradient con axis
        texto3 = Text("MANIM", font_size=72, weight=BOLD)
        texto3.set_color_by_gradient(BLUE, PURPLE)
        texto3.next_to(texto2, DOWN, buff=0.8)
        
        self.play(Write(texto1))
        self.play(Write(texto2))
        self.play(Write(texto3))
        self.wait()

---
## 3. 🔢 Indexación de Text - Acceso a caracteres

`Text` se comporta como un **VGroup** donde cada carácter es un submobject.
Esto permite acceder a caracteres individuales por índice o slicing:

```python
texto = Text("HOLA")
texto[0]     # 'H' (primer carácter)
texto[1:3]   # 'OL' (caracteres 1 y 2)
texto[-1]    # 'A' (último carácter)
```

**⚠️ Importante**: Los espacios también cuentan como caracteres.

In [ ]:
%%manim -qm -v WARNING TextIndexacion

class TextIndexacion(Scene):
    def construct(self):
        texto = Text("MANIM", font_size=96)
        self.play(Write(texto))
        self.wait(0.5)
        
        # Animar cada letra individualmente
        for i, letra in enumerate(texto):
            self.play(
                letra.animate.set_color([RED, ORANGE, YELLOW, GREEN, BLUE][i]),
                run_time=0.3
            )
        
        self.wait(0.5)
        
        # Acceder por slicing
        self.play(
            texto[0:2].animate.shift(UP),      # "MA" sube
            texto[3:].animate.shift(DOWN),     # "IM" baja
            run_time=0.8
        )
        
        self.wait(0.5)
        
        # Volver a la posición original
        self.play(
            texto[0:2].animate.shift(DOWN),
            texto[3:].animate.shift(UP),
            run_time=0.5
        )
        
        # Escalar letra individual
        self.play(texto[2].animate.scale(1.5).set_color(YELLOW))
        
        self.wait()

---
## 4. ∑ Clase MathTex - Fórmulas Matemáticas con LaTeX

`MathTex` renderiza fórmulas usando **LaTeX en modo matemático**.
Es equivalente a escribir dentro de `$...$` o `\[...\]` en LaTeX.

### Parámetros principales:
| Parámetro | Descripción |
|-----------|-------------|
| `tex_strings` | String(s) con código LaTeX |
| `tex_environment` | Entorno LaTeX (default: `align*`) |
| `substrings_to_isolate` | Lista de substrings para aislar como submobjects |
| `tex_to_color_map` | Dict para colorear partes específicas |

### Símbolos comunes:
```python
MathTex(r"\frac{a}{b}")      # Fracción
MathTex(r"\sqrt{x}")          # Raíz cuadrada
MathTex(r"\int_0^1 f(x)dx")   # Integral
MathTex(r"\sum_{i=1}^n")      # Sumatoria
MathTex(r"\alpha, \beta")     # Letras griegas
```

In [ ]:
%%manim -qm -v WARNING MathTexBasico

class MathTexBasico(Scene):
    def construct(self):
        # Ecuación cuadrática
        ecuacion = MathTex(r"x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}")
        titulo = Text("Fórmula Cuadrática", font_size=36).to_edge(UP)
        
        self.play(Write(titulo))
        self.play(Write(ecuacion))
        self.wait()
        
        # Más ejemplos de fórmulas
        formulas = VGroup(
            MathTex(r"\int_0^\infty e^{-x^2} dx = \frac{\sqrt{\pi}}{2}"),
            MathTex(r"e^{i\pi} + 1 = 0"),
            MathTex(r"\sum_{n=1}^{\infty} \frac{1}{n^2} = \frac{\pi^2}{6}"),
            MathTex(r"\nabla \cdot \vec{E} = \frac{\rho}{\epsilon_0}")
        ).arrange(DOWN, buff=0.6).scale(0.9)
        
        self.play(FadeOut(ecuacion), FadeOut(titulo))
        
        for formula in formulas:
            self.play(Write(formula), run_time=1)
        
        self.wait()

---
## 5. 🔧 Aislamiento de Partes - La Clave para Transformaciones

### ⭐ Concepto Fundamental: Doble Llave `{{}}`

Por defecto, `MathTex` agrupa caracteres de forma automática. Para controlar
**exactamente** qué partes se convierten en submobjects separados, usa **doble llave `{{}}`**:

```python
# Sin doble llave: agrupación automática (difícil de animar partes)
MathTex(r"x^2 + y^2 = z^2")

# Con doble llave: cada parte entre {{}} es un submobject separado
MathTex(r"{{x}}^2 + {{y}}^2 = {{z}}^2")
```

### ¿Por qué es importante?
- Permite **transformar partes específicas** de una ecuación
- Es **ESENCIAL** para `TransformMatchingTex`
- Facilita colorear y animar elementos individuales

In [ ]:
%%manim -qm -v WARNING DobleLlaveDemo

class DobleLlaveDemo(Scene):
    def construct(self):
        titulo = Text("Comparación: Sin vs Con doble llave", font_size=28).to_edge(UP)
        
        # Sin doble llave
        sin_llave = MathTex(r"a^2 + b^2 = c^2")
        label_sin = Text("Sin {{}}: submobjects automáticos", font_size=20, color=GRAY)
        
        # Con doble llave
        con_llave = MathTex(r"{{a}}^2 + {{b}}^2 = {{c}}^2")
        label_con = Text("Con {{}}: control total", font_size=20, color=GRAY)
        
        # Posicionar
        sin_llave.shift(UP * 1.5)
        label_sin.next_to(sin_llave, DOWN, buff=0.2)
        con_llave.shift(DOWN * 0.5)
        label_con.next_to(con_llave, DOWN, buff=0.2)
        
        self.play(Write(titulo))
        self.play(Write(sin_llave), Write(label_sin))
        self.play(Write(con_llave), Write(label_con))
        self.wait()
        
        # Mostrar número de submobjects
        info_sin = Text(f"Submobjects: {len(sin_llave)}", font_size=20, color=YELLOW)
        info_con = Text(f"Submobjects: {len(con_llave)}", font_size=20, color=GREEN)
        info_sin.next_to(sin_llave, RIGHT, buff=1)
        info_con.next_to(con_llave, RIGHT, buff=1)
        
        self.play(Write(info_sin), Write(info_con))
        self.wait()
        
        # Demostrar acceso con doble llave - colorear 'a', 'b', 'c'
        # En con_llave: [0]='a', [1]='^2', [2]='+', [3]='b', [4]='^2', [5]='=', [6]='c', [7]='^2'
        self.play(
            con_llave[0].animate.set_color(RED),    # 'a'
            con_llave[3].animate.set_color(BLUE),   # 'b'  
            con_llave[6].animate.set_color(GREEN),  # 'c'
        )
        
        demo_text = Text("¡Con {{}} podemos colorear cada variable!", font_size=24, color=YELLOW)
        demo_text.to_edge(DOWN)
        self.play(Write(demo_text))
        
        self.wait()

### 5.1 Alternativa: substrings_to_isolate

Si no quieres usar doble llave, puedes usar `substrings_to_isolate`:

```python
MathTex(r"a^2 + b^2 = c^2", substrings_to_isolate=["a", "b", "c"])
```

Esto produce el mismo efecto que la doble llave.

In [ ]:
%%manim -qm -v WARNING SubstringsToIsolate

class SubstringsToIsolate(Scene):
    def construct(self):
        # Usando substrings_to_isolate
        formula = MathTex(
            r"E = mc^2",
            substrings_to_isolate=["E", "m", "c"]
        )
        formula.scale(2)
        
        self.play(Write(formula))
        self.wait(0.5)
        
        # Ahora podemos acceder a cada parte aislada
        # get_part_by_tex retorna el submobject que coincide
        E = formula.get_part_by_tex("E")
        m = formula.get_part_by_tex("m")
        c = formula.get_part_by_tex("c")
        
        self.play(E.animate.set_color(YELLOW).scale(1.2))
        self.play(m.animate.set_color(RED).scale(1.2))
        self.play(c.animate.set_color(BLUE).scale(1.2))
        
        # Agregar etiquetas
        label_E = Text("Energía", font_size=24, color=YELLOW)
        label_m = Text("Masa", font_size=24, color=RED)
        label_c = Text("Velocidad de la luz", font_size=24, color=BLUE)
        
        label_E.next_to(E, UP)
        label_m.next_to(m, DOWN)
        label_c.next_to(c, DOWN + RIGHT)
        
        self.play(Write(label_E), Write(label_m), Write(label_c))
        
        self.wait()

---
## 6. 🎯 Métodos de MathTex para Manipulación

### Métodos principales:

| Método | Descripción |
|--------|-------------|
| `get_part_by_tex(tex)` | Retorna el primer submobject que coincide con tex |
| `get_parts_by_tex(tex)` | Retorna TODOS los submobjects que coinciden |
| `set_color_by_tex(tex, color)` | Colorea todas las coincidencias |
| `set_color_by_tex_to_color_map(dict)` | Colorea múltiples partes con un diccionario |
| `set_opacity_by_tex(tex, opacity)` | Cambia opacidad de coincidencias |

### Usando tex_to_color_map en constructor:
```python
MathTex(
    r"a^2 + b^2 = c^2",
    tex_to_color_map={"a": RED, "b": BLUE, "c": GREEN}
)
```

In [ ]:
%%manim -qm -v WARNING MetodosMathTex

class MetodosMathTex(Scene):
    def construct(self):
        # Crear fórmula con tex_to_color_map
        formula = MathTex(
            r"{{a}}^2 + {{b}}^2 = {{c}}^2",
            tex_to_color_map={"a": RED, "b": BLUE, "c": GREEN}
        ).scale(1.5)
        
        titulo = Text("tex_to_color_map en constructor", font_size=28).to_edge(UP)
        
        self.play(Write(titulo))
        self.play(Write(formula))
        self.wait()
        
        # Cambiar a nueva fórmula
        self.play(FadeOut(formula), FadeOut(titulo))
        
        # Usando set_color_by_tex después de crear
        formula2 = MathTex(
            r"{{x}} + {{y}} = {{x}} + {{y}}"
        ).scale(1.5)
        
        titulo2 = Text("set_color_by_tex (colorea TODAS las coincidencias)", font_size=24).to_edge(UP)
        
        self.play(Write(titulo2), Write(formula2))
        self.wait(0.5)
        
        # set_color_by_tex colorea TODAS las 'x' y todas las 'y'
        formula2.set_color_by_tex("x", YELLOW)
        formula2.set_color_by_tex("y", PURPLE)
        
        self.play(formula2.animate.set_color_by_tex("x", YELLOW))
        self.wait(0.3)
        self.play(formula2.animate.set_color_by_tex("y", PURPLE))
        
        self.wait()
        
        # Demostrar get_parts_by_tex (retorna todas las coincidencias)
        self.play(FadeOut(formula2), FadeOut(titulo2))
        
        formula3 = MathTex(r"{{x}} \cdot {{x}} = {{x}}^2").scale(1.5)
        titulo3 = Text("get_parts_by_tex - obtiene todas las coincidencias", font_size=24).to_edge(UP)
        
        self.play(Write(titulo3), Write(formula3))
        
        # get_parts_by_tex retorna VGroup con todas las 'x'
        todas_x = formula3.get_parts_by_tex("x")
        
        # Animar todas juntas
        self.play(todas_x.animate.set_color(ORANGE).scale(1.3))
        
        info = Text(f"Se encontraron {len(todas_x)} coincidencias de 'x'", font_size=24, color=GREEN)
        info.to_edge(DOWN)
        self.play(Write(info))
        
        self.wait()

---
## 7. 📄 Clase Tex - LaTeX en Modo Texto

`Tex` es para texto LaTeX **fuera del modo matemático**. 
Es útil para combinar texto normal con fórmulas.

### Diferencia clave:
- `MathTex`: Todo está en modo matemático (`$...$`)
- `Tex`: Modo texto normal, usa `$...$` para matemáticas inline

In [ ]:
%%manim -qm -v WARNING TexVsMathTex

class TexVsMathTex(Scene):
    def construct(self):
        titulo = Text("Tex vs MathTex", font_size=36).to_edge(UP)
        
        # Tex - modo texto (puede incluir matemáticas con $)
        tex_ejemplo = Tex(
            r"La ecuación $E = mc^2$ es famosa.",
            font_size=36
        )
        tex_label = Text("Tex (modo texto)", font_size=20, color=YELLOW)
        
        # MathTex - todo es matemáticas
        mathtex_ejemplo = MathTex(
            r"E = mc^2",
            font_size=48
        )
        mathtex_label = Text("MathTex (modo matemático)", font_size=20, color=GREEN)
        
        # Posicionar
        tex_ejemplo.shift(UP)
        tex_label.next_to(tex_ejemplo, LEFT, buff=0.5)
        mathtex_ejemplo.shift(DOWN)
        mathtex_label.next_to(mathtex_ejemplo, LEFT, buff=0.5)
        
        self.play(Write(titulo))
        self.play(Write(tex_ejemplo), Write(tex_label))
        self.play(Write(mathtex_ejemplo), Write(mathtex_label))
        
        self.wait()
        
        # Mostrar uso avanzado de Tex con múltiples partes
        self.play(FadeOut(VGroup(tex_ejemplo, tex_label, mathtex_ejemplo, mathtex_label, titulo)))
        
        # Tex con múltiples argumentos (cada uno se convierte en submobject)
        tex_multi = Tex(
            r"Sea ", r"$x$", r" un número real tal que ", r"$x > 0$"
        )
        
        self.play(Write(tex_multi))
        self.wait(0.5)
        
        # Colorear las partes matemáticas (índices 1 y 3)
        self.play(
            tex_multi[1].animate.set_color(YELLOW),
            tex_multi[3].animate.set_color(GREEN)
        )
        
        self.wait()

---
## 8. ⭐ TRANSFORMACIONES DE TEXTO - Sección Extendida

Esta es la sección más importante para crear animaciones matemáticas profesionales.
Aquí aprenderás a transformar ecuaciones de manera elegante.

### 8.1 TransformMatchingTex

`TransformMatchingTex` es la **animación más poderosa** para ecuaciones.
Transforma una fórmula LaTeX en otra, **emparejando partes que coinciden**.

#### ¿Cómo funciona?
1. Compara los `tex_string` de cada submobject
2. Las partes iguales se **transforman** entre sí
3. Las partes diferentes **aparecen/desaparecen**

#### Requisitos:
- **DEBE** usar doble llave `{{}}` para que funcione correctamente
- Los strings dentro de `{{}}` deben coincidir entre origen y destino

#### Parámetros importantes:
| Parámetro | Descripción |
|-----------|-------------|
| `transform_mismatches` | Si True, transforma partes que no coinciden (default: False) |
| `fade_transform_mismatches` | Si True, hace fade en partes que no coinciden (default: False) |
| `key_map` | Dict que mapea strings origen→destino para forzar emparejamiento |
| `path_arc` | Ángulo del arco para el movimiento (en radianes) |

In [ ]:
%%manim -qm -v WARNING TransformMatchingTexBasico

class TransformMatchingTexBasico(Scene):
    def construct(self):
        titulo = Text("TransformMatchingTex - Ejemplo Básico", font_size=28).to_edge(UP)
        
        # IMPORTANTE: Usar doble llave {{}} para cada parte que queremos emparejar
        eq1 = MathTex(r"{{a}}^2 + {{b}}^2 = {{c}}^2")
        eq2 = MathTex(r"{{a}}^2 = {{c}}^2 - {{b}}^2")
        eq3 = MathTex(r"{{a}} = \sqrt{{{c}}^2 - {{b}}^2}")
        
        eq1.scale(1.5)
        eq2.scale(1.5)
        eq3.scale(1.5)
        
        self.play(Write(titulo))
        self.play(Write(eq1))
        self.wait()
        
        # Primera transformación: despejando a²
        self.play(TransformMatchingTex(eq1, eq2))
        self.wait()
        
        # Segunda transformación: sacando raíz
        self.play(TransformMatchingTex(eq2, eq3))
        self.wait()
        
        # Nota explicativa
        nota = Text(
            "Las partes iguales (a, b, c) se transforman entre sí",
            font_size=20,
            color=YELLOW
        ).to_edge(DOWN)
        self.play(Write(nota))
        
        self.wait()

### 8.2 Usando key_map para Forzar Emparejamiento

Cuando las variables **cambian de nombre** (ej: x→a), usa `key_map` para
indicar qué parte del origen corresponde a qué parte del destino:

```python
TransformMatchingTex(
    eq1, eq2,
    key_map={"x": "a", "y": "b"}  # x se transforma en a, y en b
)
```

In [ ]:
%%manim -qm -v WARNING TransformMatchingTexKeyMap

class TransformMatchingTexKeyMap(Scene):
    def construct(self):
        titulo = Text("key_map: Sustitución de Variables", font_size=28).to_edge(UP)
        
        # Variables genéricas
        variables = VGroup(
            MathTex("a", color=RED),
            MathTex("b", color=BLUE),
            MathTex("c", color=GREEN)
        ).arrange(RIGHT, buff=1).shift(UP * 2.5)
        
        # Ecuación con x, y, z
        eq1 = MathTex(r"{{x}}^2", "+", "{{y}}^2", "=", "{{z}}^2")
        eq1.scale(1.5)
        
        # Ecuación con a, b, c (sustituimos las variables)
        eq2 = MathTex(r"{{a}}^2", "+", "{{b}}^2", "=", "{{c}}^2")
        eq2.scale(1.5)
        
        self.play(Write(titulo))
        self.play(Write(variables))
        self.play(Write(eq1))
        self.wait()
        
        # key_map indica qué variable del origen corresponde a cuál del destino
        self.play(
            TransformMatchingTex(
                Group(eq1, variables),  # Origen incluye las variables
                eq2,
                key_map={"x": "a", "y": "b", "z": "c"}  # Mapeo de sustitución
            )
        )
        
        self.wait()
        
        # Ahora reorganizar la ecuación
        eq3 = MathTex(r"{{a}}^2", "=", "{{c}}^2", "-", "{{b}}^2")
        eq3.scale(1.5)
        
        self.play(TransformMatchingTex(eq2, eq3))
        
        nota = Text("key_map fuerza x→a, y→b, z→c", font_size=22, color=YELLOW).to_edge(DOWN)
        self.play(Write(nota))
        
        self.wait()

### 8.3 transform_mismatches y fade_transform_mismatches

Controla qué hacer con las partes que **no coinciden**:

| Parámetro | Comportamiento |
|-----------|----------------|
| Ambos `False` (default) | Partes no coincidentes: origen desaparece, destino aparece |
| `transform_mismatches=True` | Transforma partes no coincidentes entre sí |
| `fade_transform_mismatches=True` | Hace FadeTransform en partes no coincidentes |

In [ ]:
%%manim -qm -v WARNING TransformMismatchesDemo

class TransformMismatchesDemo(Scene):
    def construct(self):
        # Demostrar las diferentes opciones para mismatches
        
        # ============ EJEMPLO 1: Default (sin flags) ============
        titulo1 = Text("Default: partes no coincidentes aparecen/desaparecen", 
                       font_size=22).to_edge(UP)
        
        eq1a = MathTex(r"{{x}} + {{1}}")
        eq1b = MathTex(r"{{x}} + {{2}}")  # '1' no coincide con '2'
        eq1a.scale(2)
        eq1b.scale(2)
        
        self.play(Write(titulo1))
        self.play(Write(eq1a))
        self.wait(0.5)
        
        # Default: '1' desaparece, '2' aparece, 'x' y '+' se mantienen
        self.play(TransformMatchingTex(eq1a, eq1b))
        self.wait()
        self.play(FadeOut(eq1b), FadeOut(titulo1))
        
        # ============ EJEMPLO 2: transform_mismatches=True ============
        titulo2 = Text("transform_mismatches=True: transforma '1' → '2'", 
                       font_size=22).to_edge(UP)
        
        eq2a = MathTex(r"{{x}} + {{1}}")
        eq2b = MathTex(r"{{x}} + {{2}}")
        eq2a.scale(2)
        eq2b.scale(2)
        
        self.play(Write(titulo2))
        self.play(Write(eq2a))
        self.wait(0.5)
        
        # transform_mismatches: '1' se transforma en '2'
        self.play(TransformMatchingTex(eq2a, eq2b, transform_mismatches=True))
        self.wait()
        self.play(FadeOut(eq2b), FadeOut(titulo2))
        
        # ============ EJEMPLO 3: fade_transform_mismatches=True ============
        titulo3 = Text("fade_transform_mismatches=True: fade suave", 
                       font_size=22).to_edge(UP)
        
        eq3a = MathTex(r"{{x}} + {{1}}")
        eq3b = MathTex(r"{{x}} + {{2}}")
        eq3a.scale(2)
        eq3b.scale(2)
        
        self.play(Write(titulo3))
        self.play(Write(eq3a))
        self.wait(0.5)
        
        # fade_transform_mismatches: transición suave con fade
        self.play(TransformMatchingTex(eq3a, eq3b, fade_transform_mismatches=True))
        
        self.wait()

### 8.4 TransformMatchingShapes - Para Texto Normal

Mientras `TransformMatchingTex` usa strings LaTeX para emparejar,
`TransformMatchingShapes` compara las **formas geométricas** de los caracteres.

Es ideal para:
- Objetos `Text` (no LaTeX)
- Anagramas y reordenamiento de letras
- Cuando las formas son iguales pero no el código LaTeX

In [ ]:
%%manim -qm -v WARNING TransformMatchingShapesDemo

class TransformMatchingShapesDemo(Scene):
    def construct(self):
        titulo = Text("TransformMatchingShapes - Anagramas", font_size=28).to_edge(UP)
        
        # Anagramas clásicos
        texto1 = Text("the morse code", font_size=48)
        texto2 = Text("here come dots", font_size=48)
        
        self.play(Write(titulo))
        self.play(Write(texto1))
        self.wait()
        
        # Las letras iguales se mueven a sus nuevas posiciones
        # path_arc añade un arco al movimiento para hacerlo más visual
        self.play(
            TransformMatchingShapes(texto1, texto2, path_arc=PI/2),
            run_time=2
        )
        self.wait()
        
        # Otro ejemplo
        self.play(FadeOut(texto2))
        
        texto3 = Text("LISTEN", font_size=72)
        texto4 = Text("SILENT", font_size=72)
        
        self.play(Write(texto3))
        self.wait()
        
        self.play(
            TransformMatchingShapes(texto3, texto4, path_arc=-PI/2),
            run_time=2
        )
        
        nota = Text("Las letras idénticas se reordenan", font_size=22, color=YELLOW).to_edge(DOWN)
        self.play(Write(nota))
        
        self.wait()

### 8.5 ReplacementTransform vs Transform

Además de `TransformMatchingTex`, existen otras transformaciones importantes:

| Animación | Comportamiento |
|-----------|----------------|
| `Transform(a, b)` | `a` se convierte en `b`, pero `a` sigue en la escena |
| `ReplacementTransform(a, b)` | `a` se convierte en `b`, y `b` reemplaza a `a` en la escena |

**Diferencia clave**: Con `Transform`, si luego haces otra animación sobre el objeto, 
usas el objeto original `a`. Con `ReplacementTransform`, debes usar `b`.

In [ ]:
%%manim -qm -v WARNING ReplacementTransformDemo

class ReplacementTransformDemo(Scene):
    def construct(self):
        # ============ Transform ============
        titulo1 = Text("Transform: el objeto original permanece", font_size=24).to_edge(UP)
        
        eq1 = MathTex(r"x", font_size=96)
        eq2 = MathTex(r"x^2", font_size=96)
        eq3 = MathTex(r"x^3", font_size=96)
        
        self.play(Write(titulo1))
        self.play(Write(eq1))
        self.wait(0.5)
        
        # Con Transform, eq1 ahora se ve como eq2, pero sigue siendo eq1
        self.play(Transform(eq1, eq2))
        self.wait(0.3)
        
        # Para seguir transformando, usamos eq1 (no eq2)
        self.play(Transform(eq1, eq3))  # Correcto: usamos eq1
        
        nota1 = Text("Siempre usamos eq1 en las animaciones", font_size=20, color=YELLOW)
        nota1.to_edge(DOWN)
        self.play(Write(nota1))
        self.wait()
        
        self.play(FadeOut(eq1), FadeOut(titulo1), FadeOut(nota1))
        
        # ============ ReplacementTransform ============
        titulo2 = Text("ReplacementTransform: el objeto se reemplaza", font_size=24).to_edge(UP)
        
        eq_a = MathTex(r"y", font_size=96)
        eq_b = MathTex(r"y^2", font_size=96)
        eq_c = MathTex(r"y^3", font_size=96)
        
        self.play(Write(titulo2))
        self.play(Write(eq_a))
        self.wait(0.5)
        
        # Con ReplacementTransform, eq_a es reemplazado por eq_b
        self.play(ReplacementTransform(eq_a, eq_b))
        self.wait(0.3)
        
        # Ahora debemos usar eq_b para la siguiente transformación
        self.play(ReplacementTransform(eq_b, eq_c))  # Correcto: usamos eq_b
        
        nota2 = Text("Cambiamos de referencia: eq_a → eq_b → eq_c", font_size=20, color=GREEN)
        nota2.to_edge(DOWN)
        self.play(Write(nota2))
        
        self.wait()

### 8.6 Ejemplo Avanzado: Derivación Paso a Paso

Aquí combinamos todas las técnicas para crear una derivación matemática completa:

In [ ]:
%%manim -qm -v WARNING DerivacionCompleta

class DerivacionCompleta(Scene):
    def construct(self):
        titulo = Text("Resolviendo una ecuación cuadrática", font_size=32).to_edge(UP)
        self.play(Write(titulo))
        
        # Paso 1: Ecuación original
        paso1 = MathTex(r"{{x}}^2 + {{6}}{{x}} + {{5}} = {{0}}")
        paso1.scale(1.3)
        
        label1 = Text("Ecuación original", font_size=20, color=GRAY)
        label1.next_to(paso1, DOWN, buff=0.5)
        
        self.play(Write(paso1), Write(label1))
        self.wait()
        
        # Paso 2: Factorizar
        paso2 = MathTex(r"({{x}} + {{1}})({{x}} + {{5}}) = {{0}}")
        paso2.scale(1.3)
        
        self.play(FadeOut(label1))
        self.play(TransformMatchingTex(paso1, paso2, fade_transform_mismatches=True))
        
        label2 = Text("Factorizamos", font_size=20, color=GRAY)
        label2.next_to(paso2, DOWN, buff=0.5)
        self.play(Write(label2))
        self.wait()
        
        # Paso 3: Separar en dos ecuaciones
        paso3a = MathTex(r"{{x}} + {{1}} = {{0}}")
        paso3b = MathTex(r"{{x}} + {{5}} = {{0}}")
        paso3a.scale(1.2)
        paso3b.scale(1.2)
        
        grupo3 = VGroup(paso3a, paso3b).arrange(RIGHT, buff=2)
        
        label3 = Text("O bien...", font_size=20, color=GRAY)
        label3.move_to(ORIGIN)
        
        self.play(FadeOut(label2))
        self.play(
            TransformMatchingTex(paso2.copy(), paso3a),
            TransformMatchingTex(paso2, paso3b),
        )
        self.play(Write(label3))
        self.wait()
        
        # Paso 4: Soluciones
        sol1 = MathTex(r"{{x}} = -{{1}}")
        sol2 = MathTex(r"{{x}} = -{{5}}")
        sol1.scale(1.3).set_color(GREEN)
        sol2.scale(1.3).set_color(GREEN)
        
        grupo_sol = VGroup(sol1, sol2).arrange(RIGHT, buff=2)
        
        self.play(FadeOut(label3))
        self.play(
            TransformMatchingTex(paso3a, sol1),
            TransformMatchingTex(paso3b, sol2),
        )
        
        # Resultado final
        resultado = MathTex(r"x \in \{-1, -5\}", color=YELLOW)
        resultado.scale(1.5).to_edge(DOWN, buff=1)
        
        box = SurroundingRectangle(resultado, color=YELLOW, buff=0.2)
        
        self.play(Write(resultado), Create(box))
        
        self.wait(2)

### 8.7 Transformaciones con path_arc

El parámetro `path_arc` añade curvatura al movimiento de los objetos.
Se mide en **radianes** y define el ángulo del arco:

- `path_arc=0`: Movimiento en línea recta (default)
- `path_arc=PI/2`: Arco de 90° (cuarto de vuelta)
- `path_arc=PI`: Semicírculo
- `path_arc=-PI/2`: Arco de 90° en sentido contrario

In [ ]:
%%manim -qm -v WARNING PathArcDemo

class PathArcDemo(Scene):
    def construct(self):
        titulo = Text("path_arc: Curvatura en transformaciones", font_size=28).to_edge(UP)
        self.play(Write(titulo))
        
        # Crear ecuaciones
        eq1 = MathTex(r"{{a}} + {{b}} = {{c}}")
        eq2 = MathTex(r"{{a}} = {{c}} - {{b}}")
        
        eq1.scale(1.5)
        eq2.scale(1.5)
        
        # Sin path_arc (línea recta)
        label1 = Text("path_arc=0 (default)", font_size=20, color=YELLOW).to_edge(DOWN)
        
        self.play(Write(eq1), Write(label1))
        self.wait(0.5)
        self.play(TransformMatchingTex(eq1, eq2, path_arc=0), run_time=2)
        self.wait()
        self.play(FadeOut(eq2), FadeOut(label1))
        
        # Con path_arc positivo
        eq3 = MathTex(r"{{a}} + {{b}} = {{c}}")
        eq4 = MathTex(r"{{a}} = {{c}} - {{b}}")
        eq3.scale(1.5)
        eq4.scale(1.5)
        
        label2 = Text("path_arc=PI/2 (arco 90°)", font_size=20, color=GREEN).to_edge(DOWN)
        
        self.play(Write(eq3), Write(label2))
        self.wait(0.5)
        self.play(TransformMatchingTex(eq3, eq4, path_arc=PI/2), run_time=2)
        self.wait()
        self.play(FadeOut(eq4), FadeOut(label2))
        
        # Con path_arc grande (semicírculo)
        eq5 = MathTex(r"{{a}} + {{b}} = {{c}}")
        eq6 = MathTex(r"{{a}} = {{c}} - {{b}}")
        eq5.scale(1.5)
        eq6.scale(1.5)
        
        label3 = Text("path_arc=PI (semicírculo)", font_size=20, color=BLUE).to_edge(DOWN)
        
        self.play(Write(eq5), Write(label3))
        self.wait(0.5)
        self.play(TransformMatchingTex(eq5, eq6, path_arc=PI), run_time=2)
        
        self.wait()

---
## 9. ✨ Animaciones de Creación de Texto

Manim ofrece varias animaciones para **mostrar texto**:

| Animación | Descripción |
|-----------|-------------|
| `Write` | Simula escribir a mano (para Tex/MathTex/Text) |
| `AddTextLetterByLetter` | Muestra letra por letra (solo Text) |
| `AddTextWordByWord` | Muestra palabra por palabra (solo Text) |
| `TypeWithCursor` | Como AddTextLetterByLetter pero con cursor parpadeante |
| `Create` | Dibuja el contorno gradualmente |
| `DrawBorderThenFill` | Dibuja borde y luego rellena |
| `Unwrite` | Inverso de Write (borra) |

In [ ]:
%%manim -qm -v WARNING AnimacionesCreacion

class AnimacionesCreacion(Scene):
    def construct(self):
        # Write - La más común
        titulo = Text("Animaciones de Creación", font_size=32).to_edge(UP)
        self.play(Write(titulo))
        
        # 1. Write
        texto1 = Text("Write: escribir a mano", font_size=28)
        texto1.shift(UP * 1.5)
        self.play(Write(texto1), run_time=1.5)
        self.wait(0.5)
        
        # 2. AddTextLetterByLetter
        texto2 = Text("AddTextLetterByLetter", font_size=28)
        texto2.shift(UP * 0.5)
        self.play(AddTextLetterByLetter(texto2, time_per_char=0.05))
        self.wait(0.5)
        
        # 3. AddTextWordByWord
        texto3 = Text("Add Text Word By Word", font_size=28)
        texto3.shift(DOWN * 0.5)
        self.play(AddTextWordByWord(texto3, time_per_char=0.1))
        self.wait(0.5)
        
        # 4. DrawBorderThenFill
        texto4 = Text("DrawBorderThenFill", font_size=28)
        texto4.shift(DOWN * 1.5)
        self.play(DrawBorderThenFill(texto4), run_time=1.5)
        self.wait(0.5)
        
        # Limpiar y mostrar Unwrite
        self.play(
            Unwrite(texto1),
            Unwrite(texto2),
            Unwrite(texto3),
            Unwrite(texto4),
            run_time=1
        )
        
        # TypeWithCursor (simula tipeo con cursor)
        texto5 = Text("Escribiendo con cursor...", font_size=36)
        cursor = Rectangle(
            width=0.1, height=0.5,
            fill_color=WHITE, fill_opacity=1,
            stroke_width=0
        )
        
        self.play(TypeWithCursor(texto5, cursor), run_time=2)
        
        self.wait()

---
## 10. 🎬 Ejemplos Prácticos Combinados

### 10.1 Demostración del Teorema de Pitágoras

In [ ]:
%%manim -qm -v WARNING TeoremaPitagoras

class TeoremaPitagoras(Scene):
    def construct(self):
        # Título
        titulo = Text("Teorema de Pitágoras", font_size=40, color=YELLOW)
        titulo.to_edge(UP)
        
        # Triángulo rectángulo
        triangulo = Polygon(
            [-2, -1.5, 0], [2, -1.5, 0], [-2, 1.5, 0],
            color=WHITE, stroke_width=3
        )
        triangulo.shift(LEFT * 2)
        
        # Etiquetas de los lados
        a_label = MathTex("a", color=RED).next_to(triangulo, LEFT, buff=0.2)
        b_label = MathTex("b", color=BLUE).next_to(triangulo, DOWN, buff=0.2)
        c_label = MathTex("c", color=GREEN).move_to(triangulo.get_center() + RIGHT * 0.8 + UP * 0.3)
        
        # Ángulo recto
        angulo_recto = Square(side_length=0.3, color=WHITE)
        angulo_recto.move_to(triangulo.get_vertices()[0] + RIGHT * 0.15 + UP * 0.15)
        
        self.play(Write(titulo))
        self.play(Create(triangulo), Create(angulo_recto))
        self.play(Write(a_label), Write(b_label), Write(c_label))
        self.wait()
        
        # Ecuación paso a paso
        eq1 = MathTex(r"{{a}}^2 + {{b}}^2 = {{c}}^2")
        eq1.scale(1.3).shift(RIGHT * 3)
        eq1.set_color_by_tex("a", RED)
        eq1.set_color_by_tex("b", BLUE)
        eq1.set_color_by_tex("c", GREEN)
        
        self.play(Write(eq1))
        self.wait()
        
        # Sustituir valores
        texto_vals = Text("Si a=3, b=4:", font_size=24)
        texto_vals.next_to(eq1, UP, buff=0.5)
        
        eq2 = MathTex(r"{{3}}^2 + {{4}}^2 = {{c}}^2")
        eq2.scale(1.3).shift(RIGHT * 3)
        eq2.set_color_by_tex("3", RED)
        eq2.set_color_by_tex("4", BLUE)
        eq2.set_color_by_tex("c", GREEN)
        
        self.play(Write(texto_vals))
        self.play(TransformMatchingTex(eq1, eq2, key_map={"a": "3", "b": "4"}))
        self.wait()
        
        # Calcular
        eq3 = MathTex(r"{{9}} + {{16}} = {{c}}^2")
        eq3.scale(1.3).shift(RIGHT * 3)
        eq3.set_color_by_tex("9", RED)
        eq3.set_color_by_tex("16", BLUE)
        eq3.set_color_by_tex("c", GREEN)
        
        self.play(TransformMatchingTex(eq2, eq3, fade_transform_mismatches=True))
        self.wait()
        
        eq4 = MathTex(r"{{25}} = {{c}}^2")
        eq4.scale(1.3).shift(RIGHT * 3)
        eq4.set_color_by_tex("25", PURPLE)
        eq4.set_color_by_tex("c", GREEN)
        
        self.play(TransformMatchingTex(eq3, eq4, fade_transform_mismatches=True))
        self.wait()
        
        eq5 = MathTex(r"{{c}} = {{5}}")
        eq5.scale(1.5).shift(RIGHT * 3)
        eq5.set_color_by_tex("c", GREEN)
        eq5.set_color_by_tex("5", YELLOW)
        
        self.play(TransformMatchingTex(eq4, eq5, fade_transform_mismatches=True))
        
        # Resultado final
        box = SurroundingRectangle(eq5, color=YELLOW, buff=0.2)
        self.play(Create(box))
        
        self.wait(2)

### 10.2 Explicación de una Derivada

In [ ]:
%%manim -qm -v WARNING ExplicacionDerivada

class ExplicacionDerivada(Scene):
    def construct(self):
        # Título
        titulo = Text("Regla de la Potencia", font_size=36).to_edge(UP)
        self.play(Write(titulo))
        
        # Regla general
        regla = MathTex(
            r"\frac{d}{dx}[{{x}}^{{n}}] = {{n}} \cdot {{x}}^{{{n}}-1}"
        )
        regla.scale(1.3)
        
        self.play(Write(regla))
        self.wait()
        
        # Colorear partes importantes
        self.play(
            regla.get_part_by_tex("n").animate.set_color(YELLOW),
        )
        
        nota = Text("n es el exponente", font_size=20, color=YELLOW)
        nota.next_to(regla, DOWN, buff=0.5)
        self.play(Write(nota))
        self.wait()
        
        self.play(FadeOut(nota), FadeOut(regla))
        
        # Ejemplo 1: x³
        ejemplo_titulo = Text("Ejemplo: f(x) = x³", font_size=28, color=GREEN)
        ejemplo_titulo.shift(UP * 2)
        
        paso1 = MathTex(r"f({{x}}) = {{x}}^{{3}}")
        paso2 = MathTex(r"f'({{x}}) = {{3}} \cdot {{x}}^{{{3}}-1}")
        paso3 = MathTex(r"f'({{x}}) = {{3}} \cdot {{x}}^{{2}}")
        paso4 = MathTex(r"f'({{x}}) = {{3}}{{x}}^{{2}}")
        
        for p in [paso1, paso2, paso3, paso4]:
            p.scale(1.3)
        
        self.play(Write(ejemplo_titulo))
        self.play(Write(paso1))
        self.wait()
        
        # Aplicar la regla
        flecha = Arrow(paso1.get_bottom(), paso2.get_top() + UP * 0.5, color=YELLOW)
        texto_regla = Text("Aplicar regla", font_size=18, color=YELLOW)
        texto_regla.next_to(flecha, RIGHT, buff=0.1)
        
        self.play(
            paso1.animate.shift(UP),
            Create(flecha),
            Write(texto_regla)
        )
        paso2.shift(DOWN * 0.5)
        
        self.play(TransformMatchingTex(paso1.copy(), paso2))
        self.wait()
        
        # Simplificar
        self.play(FadeOut(flecha), FadeOut(texto_regla), FadeOut(paso1))
        self.play(paso2.animate.move_to(ORIGIN))
        
        self.play(TransformMatchingTex(paso2, paso3, fade_transform_mismatches=True))
        self.wait()
        
        self.play(TransformMatchingTex(paso3, paso4, fade_transform_mismatches=True))
        
        # Resultado
        box = SurroundingRectangle(paso4, color=GREEN, buff=0.2)
        resultado_label = Text("Resultado", font_size=20, color=GREEN)
        resultado_label.next_to(box, DOWN)
        
        self.play(Create(box), Write(resultado_label))
        
        self.wait(2)

### 10.3 Animación de Texto Narrativo

In [ ]:
%%manim -qm -v WARNING TextoNarrativo

class TextoNarrativo(Scene):
    def construct(self):
        # Texto con diferentes estilos para contar una historia
        
        linea1 = Text("En matemáticas...", font_size=36)
        self.play(AddTextLetterByLetter(linea1, time_per_char=0.05))
        self.wait(0.5)
        self.play(linea1.animate.to_edge(UP))
        
        # Fórmula famosa
        euler = MathTex(r"e^{i\pi} + 1 = 0", font_size=72)
        euler_label = Text("La Identidad de Euler", font_size=24, color=YELLOW)
        euler_label.next_to(euler, UP)
        
        self.play(Write(euler_label))
        self.play(Write(euler), run_time=2)
        self.wait()
        
        # Descripción con Tex (mezcla texto y matemáticas)
        descripcion = Tex(
            r"Conecta cinco constantes fundamentales:",
            font_size=28
        )
        descripcion.next_to(euler, DOWN, buff=0.8)
        
        self.play(Write(descripcion))
        self.wait()
        
        # Listar las constantes
        constantes = VGroup(
            MathTex(r"e", r" \text{ (número de Euler)}", font_size=28),
            MathTex(r"i", r" \text{ (unidad imaginaria)}", font_size=28),
            MathTex(r"\pi", r" \text{ (pi)}", font_size=28),
            MathTex(r"1", r" \text{ (unidad multiplicativa)}", font_size=28),
            MathTex(r"0", r" \text{ (elemento neutro)}", font_size=28),
        ).arrange(DOWN, buff=0.3, aligned_edge=LEFT)
        
        constantes.scale(0.8).next_to(descripcion, DOWN, buff=0.5)
        
        # Colorear símbolos
        colores = [BLUE, GREEN, RED, YELLOW, PURPLE]
        for const, color in zip(constantes, colores):
            const[0].set_color(color)
        
        for const in constantes:
            self.play(Write(const), run_time=0.5)
        
        self.wait()
        
        # Final
        conclusion = Text(
            "¡La ecuación más bella de las matemáticas!",
            font_size=28,
            color=GOLD
        )
        conclusion.to_edge(DOWN)
        
        self.play(Write(conclusion))
        
        self.wait(2)

---
## 11. 📝 Ejercicios

### Ejercicio 1: Estilo de Texto
Crea un texto que diga "Python es el mejor lenguaje" donde:
- "Python" esté en azul y negrita
- "mejor" esté en verde y cursiva
- "lenguaje" tenga un gradiente rojo→naranja

### Ejercicio 2: Transformación de Ecuación
Crea una animación que transforme:
1. `ax + b = c`
2. `ax = c - b`
3. `x = (c - b) / a`

Usa `TransformMatchingTex` con doble llave para cada variable.

### Ejercicio 3: Anagrama con TransformMatchingShapes
Crea una animación que transforme "TRIANGLE" en "INTEGRAL" usando `TransformMatchingShapes`.

### Ejercicio 4: Explicación Matemática
Crea una escena que explique paso a paso cómo resolver:
`2x + 4 = 10`
Incluye texto explicativo y colorea las operaciones.

In [ ]:
# Espacio para resolver los ejercicios

# Ejercicio 1


# Ejercicio 2


# Ejercicio 3


# Ejercicio 4


---
## 12. 📚 Resumen y Referencia Rápida

### Clases principales:
```python
Text("texto")                    # Texto sin LaTeX (usa Pango)
MathTex(r"x^2")                  # LaTeX modo matemático
Tex(r"Texto con $x^2$ inline")   # LaTeX modo texto
```

### Estilizado de Text:
```python
Text("abc", t2c={"a": RED})      # text to color
Text("abc", t2w={"a": BOLD})     # text to weight
Text("abc", t2s={"a": ITALIC})   # text to slant
Text("abc", t2f={"a": "Arial"})  # text to font
```

### Aislamiento de partes en MathTex:
```python
MathTex(r"{{x}}^2 + {{y}}^2")                          # Doble llave
MathTex(r"x^2 + y^2", substrings_to_isolate=["x","y"]) # Alternativa
```

### Métodos de MathTex:
```python
formula.get_part_by_tex("x")       # Obtener parte
formula.get_parts_by_tex("x")      # Obtener todas las partes
formula.set_color_by_tex("x", RED) # Colorear
```

### Transformaciones:
```python
TransformMatchingTex(eq1, eq2)                           # Por tex_string
TransformMatchingTex(eq1, eq2, key_map={"x": "a"})       # Con mapeo
TransformMatchingTex(eq1, eq2, transform_mismatches=True) # Transforma no coincidentes
TransformMatchingTex(eq1, eq2, path_arc=PI/2)            # Con arco

TransformMatchingShapes(text1, text2)    # Por forma geométrica
ReplacementTransform(a, b)               # Reemplaza referencia
Transform(a, b)                          # Mantiene referencia original
```

### Animaciones de creación:
```python
Write(texto)                    # Escribir
AddTextLetterByLetter(texto)    # Letra por letra
AddTextWordByWord(texto)        # Palabra por palabra
TypeWithCursor(texto, cursor)   # Con cursor
Unwrite(texto)                  # Borrar
DrawBorderThenFill(texto)       # Borde y relleno
```

---
## 13. 🔗 Recursos y Referencias

### Documentación oficial:
- [Text - Manim Docs](https://docs.manim.community/en/stable/reference/manim.mobject.text.text_mobject.Text.html)
- [MathTex - Manim Docs](https://docs.manim.community/en/stable/reference/manim.mobject.text.tex_mobject.MathTex.html)
- [Tex - Manim Docs](https://docs.manim.community/en/stable/reference/manim.mobject.text.tex_mobject.Tex.html)
- [TransformMatchingTex](https://docs.manim.community/en/stable/reference/manim.animation.transform_matching_parts.TransformMatchingTex.html)
- [TransformMatchingShapes](https://docs.manim.community/en/stable/reference/manim.animation.transform_matching_parts.TransformMatchingShapes.html)
- [Creation Animations](https://docs.manim.community/en/stable/reference/manim.animation.creation.html)

### Tutoriales relacionados:
- [Writing Mathematical Formulas](https://docs.manim.community/en/stable/tutorials/quickstart.html)
- [Manim Example Gallery - Text](https://docs.manim.community/en/stable/examples.html#text-examples)

### Tips LaTeX:
- Siempre usa `r"..."` (raw string) para código LaTeX
- Usa `\\` para nueva línea en LaTeX
- Símbolos comunes: `\alpha`, `\beta`, `\gamma`, `\pi`, `\theta`
- Fracciones: `\frac{num}{den}`
- Raíces: `\sqrt{x}`, `\sqrt[n]{x}`
- Integrales: `\int_a^b`, `\iint`, `\oint`
- Sumatorias: `\sum_{i=0}^n`, `\prod`

---

**¡Felicidades!** Has completado la Lección 8 sobre Text y Tex.
En la próxima lección profundizaremos en más tipos de Transformaciones.